# 01 — Clean & Structure

First stage of the ML feature pipeline (see `ML_PLAN.md`). Loads `magicbricks_combined.xlsx` (1,001 localities, ~100 per city),
standardises missing values, parses the dense price string, counts amenities, normalises the free-text
blocks, geocodes by pincode, audits quality, and writes `artifacts/localities_clean.parquet` for the
downstream feature-engineering, NLP, geo/graph, and modelling notebooks.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

def find_file(name):
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(name)

XLSX = find_file("magicbricks_combined.xlsx")
ARTDIR = Path.cwd() / "artifacts"
ARTDIR.mkdir(exist_ok=True)
print("Reading:", XLSX)
print("Artifacts ->", ARTDIR)

Reading: C:\Users\singh\Desktop\GOATLife\magicbricks_combined.xlsx
Artifacts -> C:\Users\singh\Desktop\GOATLife\notebooks\artifacts


In [2]:
PRICE_COL = "price range(for residential, office space, shop)"
AMENITY_COLS = ["Educational Institute", "Hospital", "Shopping Centre",
                "Transportation Hub", "Commercial Hub", "Tourist Spot", "Nearby Localities"]
TEXT_COLS = ["Physical infrastructure", "Locality introduction and neighbourhood",
             "Social & retail infra", "Nearby employment hubs"]

df = pd.read_excel(XLSX, dtype={"PINCODE": "string"})
print("Original shape:", df.shape)

# Standardise 'N/A' / 'NA' / blank -> NaN (whole-cell)
df = df.replace(r"^\s*(N/?A)?\s*$", np.nan, regex=True)

Original shape: (1001, 16)


In [3]:
# Parse the dense price string -> residential buy/rent min/max + midpoints
_NUM = r"Rs\.?\s*([\d,]+)\s*-\s*Rs\.?\s*([\d,]+)"

def _to_float(s):
    return float(s.replace(",", ""))

def extract_residential_prices(price_string):
    cols = ["res_min_buy", "res_max_buy", "res_min_rent", "res_max_rent"]
    vals = {c: np.nan for c in cols}
    if pd.isna(price_string):
        return pd.Series(vals)
    segment = next((s for s in re.split(r"\|\|", str(price_string)) if "Residential" in s), "")
    buy = re.search(r"Buy\s*" + _NUM, segment)
    if buy:
        vals["res_min_buy"], vals["res_max_buy"] = _to_float(buy.group(1)), _to_float(buy.group(2))
    rent = re.search(r"Rent\s*" + _NUM, segment)
    if rent:
        vals["res_min_rent"], vals["res_max_rent"] = _to_float(rent.group(1)), _to_float(rent.group(2))
    return pd.Series(vals)

df = pd.concat([df, df[PRICE_COL].apply(extract_residential_prices)], axis=1)
df["res_avg_buy"] = df[["res_min_buy", "res_max_buy"]].mean(axis=1)
df["res_avg_rent"] = df[["res_min_rent", "res_max_rent"]].mean(axis=1)

In [4]:
# Amenity lists -> de-duplicated counts
def count_amenities(value):
    if pd.isna(value):
        return 0
    items = [p.strip() for p in str(value).split(",")]
    items = [p for p in items if p and p.upper() != "N/A"]
    return len({p.lower() for p in items})

for col in AMENITY_COLS:
    if col in df.columns:
        df["num_" + col.lower().replace(" ", "_")] = df[col].apply(count_amenities)

# Normalise free-text blocks
def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text or np.nan

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

In [5]:
# Geocode by pincode (offline, pgeocode) so downstream geo/graph notebooks have lat/lng
import pgeocode

_nomi = pgeocode.Nominatim("in")
_cache = {}

def geocode(pin):
    if pd.isna(pin):
        return (np.nan, np.nan)
    p = re.sub(r"\D", "", str(pin))[:6]
    if len(p) != 6:
        return (np.nan, np.nan)
    if p not in _cache:
        r = _nomi.query_postal_code(p)
        _cache[p] = (np.nan, np.nan) if (r is None or pd.isna(r.latitude)) else (round(float(r.latitude), 5), round(float(r.longitude), 5))
    return _cache[p]

coords = df["PINCODE"].apply(lambda p: pd.Series(geocode(p), index=["lat", "lng"]))
df = pd.concat([df, coords], axis=1)
print("Geocoded:", df["lat"].notna().sum(), "/", len(df))

Geocoded: 886 / 1001


In [6]:
# Quality audit
n = len(df)
print("Final shape:", df.shape)
print("Residential buy parsed:", df["res_min_buy"].notna().sum(), "/", n)
print("Geocoded:", df["lat"].notna().sum(), "/", n)
print()
print("Missingness (top 12 columns by missing %):")
miss = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(miss.head(12).to_string())

Final shape: (1001, 31)
Residential buy parsed: 549 / 1001
Geocoded: 886 / 1001

Missingness (top 12 columns by missing %):
Tourist Spot                                        90.6
Commercial Hub                                      71.9
Hospital                                            64.6
Transportation Hub                                  60.1
Shopping Centre                                     53.5
res_max_buy                                         45.2
res_min_buy                                         45.2
res_max_rent                                        45.2
res_min_rent                                        45.2
res_avg_buy                                         45.2
res_avg_rent                                        45.2
price range(for residential, office space, shop)    44.8


In [7]:
# Persist for the downstream notebooks
out = ARTDIR / "localities_clean.parquet"
df.to_parquet(out, index=False)
print("Saved", df.shape[0], "rows x", df.shape[1], "cols ->", out.relative_to(Path.cwd()))

Saved 1001 rows x 31 cols -> artifacts\localities_clean.parquet


## Output
`artifacts/localities_clean.parquet` — cleaned, price-parsed, amenity-counted, geocoded.
**Next:** `02_feature_engineering.ipynb` derives ratios, diversity/entropy, and interaction features.